# Black-Scholes Pricing and Payoffs

Part 1 of a four-notebook series on options pricing. This notebook covers the core Black-Scholes formulas for vanilla European calls and puts, and how an option's current value compares to its payoff at expiry.

The pricing functions themselves live in [`src/optionspricing`](../src/optionspricing) so they can be reused and unit-tested (see [`tests/`](../tests)) instead of being redefined in every notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
from optionspricing import black_scholes_call, black_scholes_put

We define a hypothetical call option with fixed parameters and graph its value against stock price. The big assumptions are that volatility and the risk-free rate are constant over the life of the option — we'll come back to how realistic that is.

In [ ]:
S = 100          # Current stock price (used only for single calculations)
K = 100          # Strike price
T = 1            # Time to expiry (years) -- tau = T - t, not the duration of the contract
r = 0.05         # Risk-free interest rate
sigma = 0.20     # Volatility

### Where this model breaks

1. **Volatility smile / skew** — BS assumes constant sigma, but real implied vols vary by strike and expiry (deep OTM puts trade at much higher implied vol than ATM). See notebooks 3 and 4.
2. **Near-expiry instability** — as T -> 0 the d1/d2 terms blow up; gamma and vega spike and delta becomes a step function. Continuous hedging is impossible.
3. **Log-normal assumption** — real returns have fat tails and negative skew. BS underestimates the probability of large moves.
4. **Constant risk-free rate** — rates are stochastic; material for long-dated options.
5. **No dividends / transaction costs** — the model assumes a frictionless world (notebook 4 adds a dividend yield term back in).

An option's value before expiry is not simply its payoff discounted to today — it also includes **time value**. Below we plot both the payoff at expiry, $Payoff(S,K) = \max(S-K,0)$ for a call, and the current Black-Scholes value across a range of stock prices.

In [ ]:
S_values = np.linspace(20, 180, 300)

call_prices = [black_scholes_call(s, K, T, r, sigma) for s in S_values]
call_payoff = np.maximum(S_values - K, 0)

plt.figure(figsize=(8, 5))
plt.plot(S_values, call_payoff, '--', label='Call Payoff at Expiry')
plt.plot(S_values, call_prices, label='Current Call Value')
plt.xlabel('Stock Price')
plt.ylabel('Value')
plt.title('Call Payoff vs Current Value')
plt.grid(True)
plt.legend()
plt.show()

**Region 1 — out of the money (S < K):** the call still has a small positive value because there remains a chance of ending in the money within a year.

**Region 2 — at the money (S ~ K):** the call is worth well above its zero intrinsic value, reflecting the probability the stock rises above K over the next year.

**Region 3 — in the money (S > K):** the option value exceeds its intrinsic value, because the exercise date is a year away, so the deferred payment of K carries time value.

The put value below can be read the same way, mirrored.

In [ ]:
put_prices = [black_scholes_put(s, K, T, r, sigma) for s in S_values]
put_payoff = np.maximum(K - S_values, 0)

plt.figure(figsize=(8, 5))
plt.plot(S_values, put_payoff, '--', label='Put Payoff at Expiry')
plt.plot(S_values, put_prices, label='Current Put Value')
plt.xlabel('Stock Price')
plt.ylabel('Value')
plt.title('Put Payoff vs Current Value')
plt.grid(True)
plt.legend()
plt.show()

Now let's drop the fixed time-to-expiry assumption and look at call value as a function of both stock price $S$ and time to maturity $\tau$. The 2D call-value plot above is just the slice of this surface at $\tau = 1$.

In [ ]:
S_grid_vals = np.linspace(20, 180, 100)
tau_vals = np.linspace(0.01, 2, 100)
S_grid, tau_grid = np.meshgrid(S_grid_vals, tau_vals)

call_surface = np.zeros_like(S_grid)
for i in range(len(tau_vals)):
    for j in range(len(S_grid_vals)):
        call_surface[i, j] = black_scholes_call(S_grid[i, j], K, tau_grid[i, j], r, sigma)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(tau_grid, S_grid, call_surface, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time to Maturity (Years)')
ax.set_ylabel('Stock Price')
ax.set_zlabel('Call Option Value')
ax.set_title('Black-Scholes Call Price Surface')
fig.colorbar(surface, shrink=0.6, aspect=10)
plt.show()

Longer time to maturity generally means a more valuable option, since there is more time for the stock to move favourably. Next: [02_greeks.ipynb](02_greeks.ipynb).